# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 927, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (329/329), done.
remote: Total 927 (delta 209), reused 331 (delta 110), pack-reused 488 (from 1)
Receiving objects: 100% (927/927), 35.14 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (410/410), done.
/content/ECE1508_GenAI


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 204.8 kB/s eta 0:00:00a 0:00:01


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 29 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  3%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  6%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 13%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 17%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 20%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 24%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [6]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

15:32:24 device: cuda
15:32:24 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
15:32:24 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
15:32:32 epoch 1/20  train_loss=0.23179  val_loss=0.11360  (3.2s)
15:32:33   -> saved best checkpoint (val_loss=0.11360) to steven/outputs/patchtst_checkpoint.pt
15:32:34 epoch 2/20  train_loss=0.16968  val_loss=0.10151  (2.0s)
15:32:34   -> saved best checkpoint (val_loss=0.10151) to steven/outputs/patchtst_checkpoint.pt
15:32:37 epoch 3/20  train_loss=0.15518  val_loss=0.09973  (2.1s)
15:32:37   -> saved best checkpoint (val_loss=0.09973) to steven/outputs/patchtst_checkpoint.pt
15:32:39 epoch 4/20  train_loss=0.14540  val_loss=0.09434  (2.1s)
15:32:39   -> saved best checkpoint (val_loss=0.09434) to steven/outputs/patchtst_checkpoint.pt
15:32:41 epoch 5/20  train_loss=0.14351  val_loss=0.09386  (1.9s)
15:32:41   -> saved best checkpoint (val_loss=0.09386) to steven/outputs/patchtst_checkpoint.

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [7]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

15:33:13 device: cuda
15:33:14 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
15:33:14 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
15:33:18 epoch 1/30  beta=0.20  train_loss=0.60433 (kl=0.8069)  val_loss=0.35800 (kl=0.8000)  (2.8s)
15:33:18   -> saved best checkpoint (val_loss=0.35800) to steven/outputs/cvae_checkpoint.pt
15:33:19 epoch 2/30  beta=0.40  train_loss=0.61934 (kl=0.8001)  val_loss=0.50890 (kl=0.8000)  (1.4s)
15:33:20 epoch 3/30  beta=0.60  train_loss=0.75386 (kl=0.8000)  val_loss=0.64209 (kl=0.8002)  (1.4s)
15:33:22 epoch 4/30  beta=0.80  train_loss=0.87453 (kl=0.8000)  val_loss=0.79526 (kl=0.8000)  (1.4s)
15:33:23 epoch 5/30  beta=1.00  train_loss=0.99792 (kl=0.8000)  val_loss=0.93872 (kl=0.8000)  (1.4s)
15:33:25 epoch 6/30  beta=1.00  train_loss=0.98262 (kl=0.8000)  val_loss=0.93038 (kl=0.8000)  (1.4s)
15:33:26 epoch 7/30  beta=1.00  train_loss=0.98047 (kl=0.8000)  val_loss=0.98361 (kl=0.8000)  (1.4s)
15:33:27

## Evaluate both models on the fixed test set

In [8]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

15:34:03 device: cuda
15:34:03 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
15:34:03 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
15:34:03 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
15:34:03 evaluating on 3000 fixed test windows
15:34:04 wrote metrics to steven/outputs/metrics.json
15:34:04 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.09809182584285736,
    0.3754827380180359
  ],
  "cvae_reparam_mae_rmse": [
    0.16711772978305817,
    0.5121148228645325
  ],
  "patchtst_ohlc_mae_rmse": [
    2.543817113247388,
    3.6663831864039174
  ],
  "cvae_ohlc_mae_rmse": [
    6.841709030393545,
    8.51539838602106
  ],
  "patchtst_volume_mae_rmse": [
    1994941.125,
    3507097.5
  ],
  "cvae_volume_mae_rmse": [
    3356200.75,
    4550187.0
  ],
  "patchtst_directional_accuracy": [
    0.533,
    0.5283333333333333,
    0.54133333333333

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [9]:
!python steven/src/update_report.py

15:34:08 updated steven/v1.md: results-samples, hit-summary, spread-summary, backtest-patchtst, backtest-cvae, buy-hold-benchmark
15:34:08 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveats in Results and Long-only backtest results, and the 'Retrain both models' checkbox under Next steps.


## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [10]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/metrics.json (deflated 81%)
  adding: steven/outputs/cvae_checkpoint.pt (deflated 9%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/samples.json (deflated 78%)
  adding: steven/outputs/sample_plots/sample2_start24602_ctx56.png (deflated 12%)
  adding: steven/outputs/sample_plots/sample3_start26125_ctx21.png (deflated 12%)
  adding: steven/outputs/sample_plots/sample4_start25376_ctx63.png (deflated 12%)
  adding: steven/outputs/sample_plots/sample0_start24754_ctx56.png (deflated 12%)
  adding: steven/outputs/sample_plots/sample1_start24729_ctx49.png (deflated 11%)
  adding: steven/outputs/patchtst_checkpoint.pt (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>